In [ ]:
!pip install transformers[sentencepiece]


In [ ]:
import transformers

In [ ]:
from transformers import pipeline

In [ ]:
import pandas as pd

questions = pd.read_csv("llm_benchmark_questions1.2.csv")

questions.head()

,id,test,domain,question,option_a,option_b,option_c,response_type,correct_answer,correct_answer_text
0,CRT7_01,CRT-7,cognitive_reflection,A bat and a ball cost $1.10 in total. The bat ...,NaN,NaN,NaN,numeric,5,5 cents
1,CRT7_02,CRT-7,cognitive_reflection,If it takes 5 machines 5 minutes to make 5 wid...,NaN,NaN,NaN,numeric,5,5 minutes
2,CRT7_03,CRT-7,cognitive_reflection,"In a lake, there is a patch of lily pads. Ever...",NaN,NaN,NaN,numeric,47,47 days
3,CRT7_04,CRT-7,cognitive_reflection,If John can drink one barrel of water in 6 day...,NaN,NaN,NaN,numeric,4,4 days
4,CRT7_05,CRT-7,cognitive_reflection,Jerry received both the 15th highest and the 1...,NaN,NaN,NaN,numeric,29,29 students


In [ ]:
crt_questions = questions[questions["test"] == "CRT-7"].copy()

crt_questions

,id,test,domain,question,option_a,option_b,option_c,response_type,correct_answer,correct_answer_text
0,CRT7_01,CRT-7,cognitive_reflection,A bat and a ball cost $1.10 in total. The bat ...,NaN,NaN,NaN,numeric,5,5 cents
1,CRT7_02,CRT-7,cognitive_reflection,If it takes 5 machines 5 minutes to make 5 wid...,NaN,NaN,NaN,numeric,5,5 minutes
2,CRT7_03,CRT-7,cognitive_reflection,"In a lake, there is a patch of lily pads. Ever...",NaN,NaN,NaN,numeric,47,47 days
3,CRT7_04,CRT-7,cognitive_reflection,If John can drink one barrel of water in 6 day...,NaN,NaN,NaN,numeric,4,4 days
4,CRT7_05,CRT-7,cognitive_reflection,Jerry received both the 15th highest and the 1...,NaN,NaN,NaN,numeric,29,29 students
5,CRT7_06,CRT-7,cognitive_reflection,"A man buys a pig for $60, sells it for $70, bu...",NaN,NaN,NaN,numeric,20,$20
6,CRT7_07,CRT-7,cognitive_reflection,"Simon decided to invest $8,000 in the stock ma...",A. broken even in the stock market,B. is ahead of where he began,C. has lost money,multiple_choice,C,C


In [44]:
model_a = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-0.5B-Instruct"
)

model_b = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct"
)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [ ]:
def ask_model(generator, row):

    question = row["question"]

    # Add options if this is a multiple-choice question
    if row["response_type"] == "multiple_choice":
        question += f"""

{row["option_a"]}
{row["option_b"]}
{row["option_c"]}
"""

    messages = [
    {
        "role": "user",
        "content": f"""
Answer the following problem.

{question}

Give the final answer in exactly the unit requested by the question.

Use this exact format:
FINAL: <answer>
"""
    }
]

    result = generator(
        messages,
        max_new_tokens=150,
        do_sample=False
    )

    return result[0]["generated_text"][-1]["content"]

In [ ]:
results = []

for _, row in crt_questions.iterrows():

    answer_a = ask_model(model_a, row)
    answer_b = ask_model(model_b, row)

    results.append({
        "id": row["id"],
        "question": row["question"],
        "correct_answer": row["correct_answer"],
        "model_a_answer": answer_a,
        "model_b_answer": answer_b
    })

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

In [ ]:
results_df = pd.DataFrame(results)

results_df

,id,question,correct_answer,model_a_answer,model_b_answer
0,CRT7_01,A bat and a ball cost $1.10 in total. The bat ...,5,Let's define the variables for the cost of the...,Let's denote the cost of the ball as \( x \) d...
1,CRT7_02,If it takes 5 machines 5 minutes to make 5 wid...,5,FINAL: 25 minutes,FINAL: 5
2,CRT7_03,"In a lake, there is a patch of lily pads. Ever...",47,FINAL: 24 days,FINAL: 47
3,CRT7_04,If John can drink one barrel of water in 6 day...,4,FINAL: 18,To determine how long it will take John and Ma...
4,CRT7_05,Jerry received both the 15th highest and the 1...,29,FINAL: There are 20 students in the class.,To determine the total number of students in t...
5,CRT7_06,"A man buys a pig for $60, sells it for $70, bu...",20,FINAL: $(100)$,"To determine how much the man has made, we nee..."
6,CRT7_07,"Simon decided to invest $8,000 in the stock ma...",C,FINAL: B. is ahead of where he began,"To determine whether Simon broke even, is ahea..."


In [45]:
results_df["correct_clean"] = (
    results_df["correct_answer"]
    .astype(str)
    .str.strip()
    .str.replace("$", "", regex=False)
    .str.upper()
)

In [48]:
results_df["model_a_final"] = results_df["model_a_answer"].apply(extract_final_answer)
results_df["model_b_final"] = results_df["model_b_answer"].apply(extract_final_answer)

In [49]:
results_df["model_a_correct"] = (
    results_df["model_a_final"] == results_df["correct_clean"]
)

results_df["model_b_correct"] = (
    results_df["model_b_final"] == results_df["correct_clean"]
)

In [50]:
score_a = results_df["model_a_correct"].sum()
score_b = results_df["model_b_correct"].sum()

print("Model A:", score_a, "/ 7")
print("Model B:", score_b, "/ 7")

Model A: 0 / 7
Model B: 2 / 7


In [ ]:
results_df.to_csv("crt7_results.csv", index=False)